# 2 Autoencoders

[**Autoencoder (2006):**](https://www.cs.toronto.edu/~hinton/absps/science.pdf) NN used for unsupervised learning. 

**Objective:** To learn a compressed, efficient representation of input data without external labels.

Instead of predicting a target label $y$ from an input $x$, an autoencoder is trained to reconstruct its own input ($x \to x'$) after passing it through a constrained internal bottleneck.

An autoencoder consists of three main components:

1. **Encoder ($f_\theta$):** NN that compresses high-dimensional input data $x$ into a lower-dimensional latent representation $z$.

$$z = f_\theta(x)$$


2. **Bottleneck (Latent Space, $z$):** Low-dimensional layer in the network to force to ignore noise and preserve most important features of data.

3. **Decoder ($g_\phi$):** NN that takes the latent *code* $z$ and attempts to reconstruct the original input as $\hat{x}$.

$$\hat{x} = g_\phi(z)$$




<p style="page-break-after:always;"></p>

### Objective functions in training

NN learns to minimize a *reconstruction* loss ($\mathcal{L}$). 

The reconstruction loss measures the difference between the input $x$ and the reconstructed output $\hat{x}$. 

**Mean Squared Error (MSE)** 

<!--
Commonly used for continuous real-valued inputs (like images normalized between 0 and 1).

For a single sample with $D$ feature dimensions (e.g., $D$ pixels in an image), the total squared reconstruction loss is:
-->

$$\mathcal{L}_{\text{MSE}}(x, \hat{x}) = \frac{1}{D} \sum_{i=1}^{D} (x_i - \hat{x}_i)^2$$

where

* $x_i$ is the original value of feature $i$.
* $\hat{x}_i$ is the reconstructed value of feature $i$ (produced by the decoder).
* $D$ is the total number of input dimensions/features.

**Binary Cross-Entropy (BCE)** 

<!--
Often used when inputs are treated as Bernoulli distributions or binary values.

In autoencoders, Binary Cross-Entropy (BCE) measures the difference between the true input values $x$ and the reconstructed output values $\hat{x}$. It is typically used when the input data features are normalized to a continuous range of $[0, 1]$ (or are binary) and treated as probabilities. For a single sample with $D$ feature dimensions (e.g., $D$ pixels in an image), the Binary Cross-Entropy loss is:
-->

$$\mathcal{L}_{\text{BCE}}(x, \hat{x}) = -\sum_{i=1}^{D} \left[ x_i \log(\hat{x}_i) + (1 - x_i) \log(1 - \hat{x}_i) \right]$$

where

* $x_i \in [0, 1]$ is the original value of feature $i$.
* $\hat{x}_i \in (0, 1)$ is the reconstructed value of feature $i$ (usually produced by a sigmoid output layer).
* $D$ is the total number of input dimensions/features.



<p style="page-break-after:always;"></p>

### Variants

**Undercomplete AE**

Latent dimension is smaller than input dimension for basic dimensionality reduction and compression.

**[Denoising AE (DAE)](https://www.cs.toronto.edu/~larocheh/publications/icml-2008-denoising-autoencoders.pdf)**

Adds noise to inputs during training to prevent learning an identity mapping

The decoder is required to reconstruct the original, uncorrupted data learning to strip noise away

*Objective*: Noise removal and robust feature extraction

*Corruption:* An original input $x$ is stochastically corrupted using a corruption distribution $q(\tilde{x} \vert{} x)$ to produce a noisy input $\tilde{x}$. Common corruption strategies

* Additive Gaussian Noise: Adds random noise: $\tilde{x} = x + \epsilon$, where $\epsilon \sim \mathcal{N}(0, \sigma^2 I)$

* Masking: Randomly sets a fraction $\nu$ of input elements to 0 (or min/max values)

* Dropout Noise: Randomly zeros out features with probability $p$ during the forward pass

<p style="page-break-after:always;"></p>

**[Sparse AE](https://web.stanford.edu/class/cs294a/sparseAutoencoder_2011new.pdf)**

Forces hidden units to be mostly inactive (close to zero) via regularization penalties for any given input

*Objective*: Feature extraction without shrinking layer size

<!--
Unlike undercomplete autoencoders, a sparse autoencoder can actually have an overcomplete latent dimension (larger than the input dimension)

The sparsity constraint ensures that even with a large capacity, the network learns meaningful, disentangled representations rather than trivial identity mappings
-->

*Steps:*

* Forward pass: Input $x$ is passed through the encoder to obtain hidden activation vector $h = f_\theta(x)$

* Activation tracking: Across a mini-batch of N samples, the average activation $\hat{\rho}_j$ of each hidden neuron $j$ is calculated as

$$\hat{\rho}_j = \frac{1}{N} \sum_{i=1}^{N} a_j(x^{(i)})$$

where $a_j(x^{(i)})$ is the activation value of hidden unit $j$ when processing input $x^{(i)}$ (i.e. sigmoid in $(0, 1)$).


* Sparsity Penalty: A penalty term is added to the loss function that penalizes any hidden unit whose average activation deviates from a small target threshold $\rho$ (e.g., $\rho = 0.05$, meaning neurons should be active only 5% of the time)

$$\mathcal{L}_{\text{SAE}}(\theta, \phi) = \mathcal{L}(x, \hat{x}) + \beta \sum_{j=1}^{K} \text{Penalty}(\rho, \hat{\rho}_j) + \lambda \Omega(W)$$

where

* $\mathcal{L}$ is MSE or BCE loss function.
* $K$ is the total number of hidden units in the latent layer.
* $\rho$ is the target sparsity parameter (a value close to $0$).
* $\hat{\rho}_j$ is the empirical average activation of neuron $j$.
* $\beta$ is a hyperparameter controlling the weight of the sparsity penalty.
* $\lambda \Omega(W)$ is an optional weight decay regularizer ($L_2$ norm) to prevent overfitting.

*Sparsity penalty functions*

* Kullback-Leibler (KL) divergence: difference between two Bernoulli distributions, one with mean $\rho$ (target) and one with mean $\hat{\rho}_j$ (actual):

$$\text{KL}(\rho \parallel \hat{\rho}_j) = \rho \log \left( \frac{\rho}{\hat{\rho}_j} \right) + (1 - \rho) \log \left( \frac{1 - \rho}{1 - \hat{\rho}_j} \right)$$

* $L_1$ regularization penalty can be directly applied to the latent activations $h$:

$$\text{L1}(h) = \sum_{j=1}^{K} \vert{}h_j\vert{}$$

*Key Benefits*

* Feature interpretability: Each hidden unit specializes in detecting a distinct, highly specific subfeature

* Overcomplete capacities: Allows the latent dimension to be larger than the input dimension ($K > D$) without overfitting or degenerating into an identity copy operation.

* Modern LLM interpretability: Modern mechanistic interpretability uses massive sparse autoencoders to extract human-understandable concepts from the hidden activations of Transformer models.

<p style="page-break-after:always;"></p>

**[Variational AE](https://arxiv.org/pdf/1312.6114)** 

Unlike conventional autoencoders that map input data $x$ to a single deterministic point in latent space, a Variational AE (VAE) maps $x$ into a probability distribution over the latent space $z$. 

This allows VAE to generate entirely new samples by sampling random vectors $z$ from a prior distribution (such as a standard normal distribution $\mathcal{N}(0, I)$) and passing them through the decoder.

**Objective:** Maximize the marginal log-likelihood of observed data $\log p_\theta(x)$

$$
\hat{\theta} = \operatorname{argmax}_\theta \log p_\theta(x)
$$

Because $x$ depends on an unobserved (latent) variable $z$, the marginal probability is defined by integrating over all possible latent values $z$

$$
p_\theta(x) = \int p_\theta(x, z) \, dz = \int p_\theta(z \mid x) \, p_\theta(x) \, dz = \mathbb{E}_{p_\theta(z\vert{}x)} \left[ \log p_\theta(x) \right]
$$

Evaluating this integral directly is computationally intractable for complex neural networks. 

Let us introduce an inference network (encoder) $q_\phi(z\vert{}x)$ to approximate the true posterior $p_\theta(z\vert{}x)$

$$
\log p_\theta(x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x) \right]
$$

Using the definition of conditional probability, we can rewrite the joint distribution $p_\theta(x, z)$ as

$$p_\theta(x) = \frac{p_\theta(x, z)}{p_\theta(z\vert{}x)}$$

Substituting this expression into the expectation

$$\log p_\theta(x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{p_\theta(z\vert{}x)} \right) \right]$$


Multiplying the numerator and denominator inside the logarithm by $q_\phi(z\vert{}x)$

$$\begin{aligned}
\log p_\theta(x) 
&= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{p_\theta(z\vert{}x)} \cdot \frac{q_\phi(z\vert{}x)}{q_\phi(z\vert{}x)} \right) \right]\\
&= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \cdot \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right]\\
&= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right) + \log \left( \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right]\\
&= \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right) \right]}_{\text{Term 1: ELBO}} + \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right]}_{\text{Term 2: KL Divergence}}
\end{aligned}$$


Term 1 is defined as the Evidence Lower Bound (ELBO), denoted as $\mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$.

By definition, the Kullback-Leibler (KL) divergence between two continuous distributions $q(z)$ and $p(z)$ is

$$D_{\text{KL}}(q \parallel p) = \int q(z) \log \left( \frac{q(z)}{p(z)} \right) dz = \mathbb{E}_{q} \left[ \log \left( \frac{q(z)}{p(z)} \right) \right]$$

Therefore, Term 2 is precisely the KL divergence between our approximate posterior $q_\phi(z\vert{}x)$ and the true posterior $p_\theta(z\vert{}x)$

$$\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right] = D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right)$$

Putting both terms back together gives the fundamental identity

$$\log p_\theta(x) = \mathcal{L}_{\text{ELBO}}(\theta, \phi; x) + D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right)$$


<p style="page-break-after:always;"></p>

The goal is to maximize the marginal log-likelihood of observed data $\log p_\theta(x)$

$$\begin{aligned}
\log p_\theta(x) &= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x) \right] \\
 &= \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right]}_{\text{ELBO}(\theta, \phi; x)} + \underbrace{D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right)}_{\ge 0}
\end{aligned}$$

Since the Kullback-Leibler (KL) divergence is always non-negative ($D_{\text{KL}} \ge 0$), the ELBO serves as a lower bound on data log-likelihood

$$\log p_\theta(x) \ge \mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$$

As the optimization of $p_\theta(x)$ is computationally intractable, instead $\mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$ is maximized 

$$\begin{aligned}
\left(\hat{\theta},\, \hat{\phi}\right) 
&= \operatorname{argmax}_{\left (\theta,\, \phi\right)} \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right]\\
&= \operatorname{argmax}_{\left (\theta,\, \phi\right)} \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x\vert{}z) p(z)}{q_\phi(z\vert{}x)} \right) \right] \\ 
&= \operatorname{argmax}_{\left (\theta,\, \phi\right)} \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) + \log \left( \frac{p(z)}{q_\phi(z\vert{}x)} \right) \right] \\ 
&= \operatorname{argmax}_{\left (\theta,\, \phi\right)} \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] + \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p(z)}{q_\phi(z\vert{}x)} \right) \right] \\
&= \operatorname{argmax}_{\left (\theta,\, \phi\right)} \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] - \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{q_\phi(z\vert{}x)}{p(z)} \right) \right] \\
&= \operatorname{argmax}_{\left (\theta,\, \phi\right)}  \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] - 
D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p(z) \right)
\end{aligned}$$

<p style="page-break-after:always;"></p>

To recap

$$
\left(\hat{\theta},\, \hat{\phi}\right) 
= \operatorname{argmax}_{\left (\theta,\, \phi\right)}  \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] - 
D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p(z) \right)
$$

where  

* $\mathbb{E}_{q_\phi(z\vert{}x)} [\log p_\theta(x\Vert{}z)]$ is the reconstruction loss that measures how accurately the decoder $p_\theta(x\Vert{}z)$ reconstructs $x$ from latent code $z$. This can be computed via MSE or BCE.

* $D_{\text{KL}}(q_\phi(z\Vert{}x) \parallel p(z))$ is the 
KL distance (regularization) that measures how much the encoded distribution $q_\phi(z\Vert{}x)$ deviates from the prior $p(z) = \mathcal{N}(0, I)$. This forces the latent space to stay centered at the origin.

<!--

Closed-Form Solution for Gaussian Latent Variables

Assuming $p(z) = \mathcal{N}(0, I)$ and $q_\phi(z\vert{}x) = \mathcal{N}(\mu_x, \text{diag}(\sigma_x^2))$, the KL divergence term can be solved analytically:

$$D_{\text{KL}}\left( \mathcal{N}(\mu_x, \sigma_x^2) \parallel \mathcal{N}(0, I) \right) = -\frac{1}{2} \sum_{j=1}^{J} \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$

Where $J$ is the number of latent dimensions, and $\mu_j, \sigma_j$ are the predicted mean and standard deviation for latent dimension $j$.

-->

Since neural networks minimize loss via gradient descent, we negate the ELBO to get our final training objective:

$$\text{Loss}_{\text{VAE}}(\theta, \phi; x) = -\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] + D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p(z) \right)$$


<p style="page-break-after:always;"></p>

*The Reparameterization Trick*

During backpropagation, gradients must flow through the latent layer $z \sim q_\phi(z\vert{}x)$

Sampling directly introduces a non-differentiable stochastic node.

[Kingma & Welling introduced the reparameterization trick](https://arxiv.org/pdf/1312.6114), propose expressing sampling deterministically by isolating the randomness into an auxiliary noise variable $\epsilon$

$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \epsilon$$

where $\epsilon \sim \mathcal{N}(0, I)$ and $\odot$ denotes element-wise multiplication.

Stochasticity is isolated inside $\epsilon$, gradients can flow through $\mu_\phi(x)$ and $\sigma_\phi(x)$.


<p style="page-break-after:always;"></p>